In [1]:
from googletrans import Translator
import re

class LanguageTranslator:
    def __init__(self):
        self.translator = Translator()
        
        # Common Indian languages and their codes
        self.indian_languages = {
            'hindi': 'hi', 'bengali': 'bn', 'telugu': 'te', 'marathi': 'mr',
            'tamil': 'ta', 'urdu': 'ur', 'gujarati': 'gu', 'kannada': 'kn',
            'malayalam': 'ml', 'punjabi': 'pa'
        }
        
    def detect_language(self, text):
        try:
            detection = self.translator.detect(text)
            return detection.lang
        except Exception as e:
            print(f"Error detecting language: {e}")
            return 'en'  # Default to English if detection fails
    
    def translate_to_english(self, text, source_language=None):
        try:
            if not source_language:
                source_language = self.detect_language(text)
            
            if source_language == 'en':
                return text
            
            translation = self.translator.translate(text, src=source_language, dest='en')
            return translation.text
        except Exception as e:
            print(f"Error translating to English: {e}")
            return text  # Return original text if translation fails
    
    def translate_from_english(self, text, target_language):
        try:
            if target_language == 'en':
                return text
            
            translation = self.translator.translate(text, src='en', dest=target_language)
            return translation.text
        except Exception as e:
            print(f"Error translating from English: {e}")
            return text  # Return original text if translation fails

In [2]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')

class IntentExtractor:
    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        
        # Patterns for different parameters
        self.area_pattern = r'\b(\d+(?:\.\d+)?)\s*(?:acre|hectare|bigha|gunta|cent|sq ft|square feet|square meter)\b'
        self.investment_pattern = r'\b(?:rs|inr|₹)?\s*(\d+(?:,\d+)*(?:\.\d+)?)\s*(?:rupees|rs|inr|₹)?\b'
        self.month_pattern = r'\b(january|february|march|april|may|june|july|august|september|october|november|december|jan|feb|mar|apr|jun|jul|aug|sep|oct|nov|dec)\b'
        
        # Location patterns - can be expanded with more specific Indian locations
        self.location_keywords = [
            'andhra', 'telangana', 'karnataka', 'tamil nadu', 'kerala', 'maharashtra', 
            'punjab', 'haryana', 'uttar pradesh', 'madhya pradesh', 'gujarat', 'rajasthan',
            'bihar', 'west bengal', 'assam', 'odisha', 'jharkhand', 'chhattisgarh'
        ]
    
    def preprocess_text(self, text):
        text = text.lower()
        tokens = word_tokenize(text)
        tokens = [word for word in tokens if word not in self.stop_words]
        return ' '.join(tokens)
    
    def extract_area(self, text):
        match = re.search(self.area_pattern, text.lower())
        if match:
            return float(match.group(1))
        return None
    
    def extract_investment(self, text):
        match = re.search(self.investment_pattern, text.lower())
        if match:
            # Remove commas from the number
            amount = match.group(1).replace(',', '')
            return float(amount)
        return None
    
    def extract_month(self, text):
        match = re.search(self.month_pattern, text.lower())
        if match:
            month = match.group(1)
            # Standardize abbreviated months
            month_mapping = {
                'jan': 'january', 'feb': 'february', 'mar': 'march', 'apr': 'april',
                'jun': 'june', 'jul': 'july', 'aug': 'august', 'sep': 'september',
                'oct': 'october', 'nov': 'november', 'dec': 'december'
            }
            return month_mapping.get(month, month)
        return None
    
    def extract_location(self, text):
        text_lower = text.lower()
        for location in self.location_keywords:
            if location in text_lower:
                return location
        return None
    
    def extract_all_parameters(self, text):
        params = {
            'area': self.extract_area(text),
            'investment': self.extract_investment(text),
            'month': self.extract_month(text),
            'location': self.extract_location(text)
        }
        return params

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [3]:
import pickle
import pandas as pd
import numpy as np

class AgriculturalChatbot:
    def __init__(self):
        self.translator = LanguageTranslator()
        self.intent_extractor = IntentExtractor()
        
        # Load ML models
        try:
            with open(r'C:\Users\User\Desktop\FINAL YEAR PROJECT\Model Train\Suggestion\crop_recommendation_model.pkl', 'rb') as file:
                self.crop_model = pickle.load(file)
            
            with open(r'C:\Users\User\Desktop\FINAL YEAR PROJECT\Model Train\Prediction\roi_prediction_model.pkl', 'rb') as file:
                self.roi_model = pickle.load(file)
                
            # Load any required encoders or scalers
            # [...]
                
            self.models_loaded = True
        except Exception as e:
            print(f"Error loading models: {e}")
            self.models_loaded = False
    
    def process_query(self, user_input, user_language=None):
        # Detect language if not provided
        if not user_language:
            user_language = self.translator.detect_language(user_input)
        
        # Translate input to English if needed
        if user_language != 'en':
            english_input = self.translator.translate_to_english(user_input, user_language)
        else:
            english_input = user_input
            
        # Extract parameters from input
        params = self.intent_extractor.extract_all_parameters(english_input)
        
        # Check if we have all required parameters
        missing_params = []
        for param, value in params.items():
            if value is None:
                missing_params.append(param)
        
        # If parameters are missing, ask for them
        if missing_params:
            response = "Please provide the following information to get accurate recommendations: " + ", ".join(missing_params)
            
            # Translate response back to user language
            if user_language != 'en':
                response = self.translator.translate_from_english(response, user_language)
            return response
        
        # If all parameters are available, make predictions
        if self.models_loaded:
            try:
                # Prepare data for crop recommendation model
                # This is simplified and needs to be adjusted based on your actual model requirements
                n, p, k = 50, 50, 50  # Default values, ideally these would come from soil data
                humidity = 50  # Default value
                ph = 7.0  # Default value
                rainfall = 200  # Default value based on location and month
                
                # You would need a database or API to get actual soil and climate data based on location
                temp = self.get_temperature_for_month(params['month'], params['location'])
                
                crop_input = np.array([[n, p, k, temp, humidity, ph, rainfall]])
                recommended_crop = self.crop_model.predict(crop_input)[0]
                
                # Prepare data for ROI prediction model
                # This would need to be adjusted based on your actual model
                crop_encoded = self.encode_crop(recommended_crop)  # You'd need to create this method
                location_encoded = self.encode_location(params['location'])  # You'd need to create this method
                month_encoded = self.encode_month(params['month'])  # You'd need to create this method
                
                roi_input = np.array([[crop_encoded, params['area'], location_encoded, params['investment'], month_encoded]])
                predicted_roi = self.roi_model.predict(roi_input)[0]
                
                # Generate response
                response = f"Based on your inputs, I recommend growing {recommended_crop}. "
                response += f"The estimated return on investment is approximately {predicted_roi:.2f}%. "
                
                # Additional advice based on crop
                response += self.get_crop_advice(recommended_crop, params['month'])
                
                # Translate response back to user language
                if user_language != 'en':
                    response = self.translator.translate_from_english(response, user_language)
                return response
                
            except Exception as e:
                print(f"Error making predictions: {e}")
                response = "I'm sorry, but I couldn't process your request due to a technical issue."
                
                # Translate response back to user language
                if user_language != 'en':
                    response = self.translator.translate_from_english(response, user_language)
                return response
        else:
            response = "The recommendation models are not available at the moment. Please try again later."
            
            # Translate response back to user language
            if user_language != 'en':
                response = self.translator.translate_from_english(response, user_language)
            return response
    
    def get_temperature_for_month(self, month, location):
        # This would be replaced with an actual database lookup or API call
        # to get historical average temperatures for the given month and location
        temperature_data = {
            'january': {'punjab': 15, 'karnataka': 25},
            'february': {'punjab': 18, 'karnataka': 26},
            # ... more months and locations
        }
        
        try:
            return temperature_data[month.lower()][location.lower()]
        except KeyError:
            return 25  # Default temperature if not found
    
    def get_crop_advice(self, crop, month):
        # This would provide specific advice for the recommended crop
        crop_advice = {
            'rice': "For rice cultivation, ensure proper irrigation and control weeds effectively.",
            'wheat': "Wheat grows best with regular watering but avoid waterlogging.",
            'maize': "Maize requires consistent moisture during the growing period.",
            # ... more crops
        }
        
        return crop_advice.get(crop.lower(), "Make sure to consult with local agricultural experts for specific advice.")
    
    # You would need to implement these encoding methods based on your model
    def encode_crop(self, crop):
        # This would convert crop name to encoded value used by the ROI model
        crop_mapping = {'rice': 0, 'wheat': 1, 'maize': 2}  # Expand this based on your data
        return crop_mapping.get(crop.lower(), 0)
    
    def encode_location(self, location):
        # This would convert location to encoded value used by the ROI model
        location_mapping = {'punjab': 0, 'karnataka': 1}  # Expand this based on your data
        return location_mapping.get(location.lower(), 0)
    
    def encode_month(self, month):
        # This would convert month to encoded value used by the ROI model
        month_mapping = {
            'january': 0, 'february': 1, 'march': 2, 'april': 3, 
            'may': 4, 'june': 5, 'july': 6, 'august': 7, 
            'september': 8, 'october': 9, 'november': 10, 'december': 11
        }
        return month_mapping.get(month.lower(), 0)